[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/Python_Functions_Classes.ipynb)

# Functions, Classes, and Lambdas

How to organize Python code for AI and Data Engineering — from simple functions to the class patterns behind sklearn, FastAPI, and Pydantic.

**Concept chapter:** [Functions and Classes](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/python/05_Control_Flow_Functions.md)

**Community:** [Delivery Momentum on Skool](https://www.skool.com/deliverymomentum)

## 1. Defining Functions

Functions in Python use `def`, indentation for the body, and optional type hints. No curly braces, no semicolons.

In [ ]:
# WHAT: Define a function with arguments and a default value.
# WHY: Reusable transforms are the building blocks of every pipeline.

def classify_score(score, threshold=0.5):
    """Classify a prediction score as positive or negative."""
    if score >= threshold:
        return "positive"
    return "negative"

# Call with positional arg
print(classify_score(0.85))

# Call with keyword arg to override default
print(classify_score(0.85, threshold=0.90))

# Apply to a batch
scores = [0.92, 0.45, 0.88, 0.31]
labels = [classify_score(s) for s in scores]
print(f"Labels: {labels}")

# You Should See:
# positive
# negative
# Labels: ['positive', 'negative', 'positive', 'negative']

In [ ]:
# WHAT: Return multiple values from a function.
# WHY: ML evaluation functions return multiple metrics at once.
# Python returns them as a tuple — unpack with multiple assignment.

def evaluate_model(y_true, y_pred):
    """Compute accuracy, precision-like, and count metrics."""
    total = len(y_true)
    correct = sum(1 for t, p in zip(y_true, y_pred) if t == p)
    accuracy = correct / total
    errors = total - correct
    return accuracy, errors, total   # returns a tuple

y_true = [1, 0, 1, 1, 0, 1, 0, 0]
y_pred = [1, 0, 1, 0, 0, 1, 1, 0]

# Unpack the tuple
acc, err, n = evaluate_model(y_true, y_pred)
print(f"Accuracy: {acc:.2%} ({err} errors out of {n})")

# You Should See:
# Accuracy: 75.00% (2 errors out of 8)

In [ ]:
# WHAT: Use *args and **kwargs for flexible functions.
# WHY: Library wrappers and logging functions need to accept
# arbitrary arguments. sklearn's Pipeline uses **kwargs heavily.

def log_event(event_type, *tags, **metadata):
    """Log a pipeline event with optional tags and metadata."""
    print(f"[{event_type.upper()}]")
    if tags:
        print(f"  Tags: {', '.join(tags)}")
    for key, value in metadata.items():
        print(f"  {key}: {value}")

# *args captures positional args as a tuple
# **kwargs captures keyword args as a dict
log_event("model_trained", "production", "v2",
          model="xgboost", accuracy=0.94, duration_sec=120)

# You Should See:
# [MODEL_TRAINED]
#   Tags: production, v2
#   model: xgboost
#   accuracy: 0.94
#   duration_sec: 120

## 2. Type Hints

Type hints document expected types. They do NOT enforce at runtime (Python is still dynamically typed), but IDEs and linters use them to catch bugs early.

In [ ]:
# WHAT: Add type hints to function signatures.
# WHY: Makes code self-documenting. FastAPI uses hints to
# auto-generate API docs. Pydantic uses them for validation.

def compute_metrics(
    predictions: list[dict],
    threshold: float = 0.5
) -> dict[str, float]:
    """Compute summary metrics from a batch of predictions."""
    scores = [p["score"] for p in predictions]
    above = [s for s in scores if s >= threshold]
    return {
        "mean_score": sum(scores) / len(scores),
        "positive_rate": len(above) / len(scores),
        "count": float(len(scores))
    }

preds = [
    {"id": 1, "score": 0.95},
    {"id": 2, "score": 0.42},
    {"id": 3, "score": 0.88}
]

metrics = compute_metrics(preds)
for name, value in metrics.items():
    print(f"  {name}: {value:.3f}")

# You Should See:
#   mean_score: 0.750
#   positive_rate: 0.667
#   count: 3.000

## 3. Lambda Functions

Lambdas are anonymous one-line functions. Use them with `sorted()`, `map()`, `filter()` — not for complex logic.

In [ ]:
# WHAT: Use lambdas with sorted(), map(), and filter().
# WHY: Pandas .apply(), sklearn's FunctionTransformer, and
# sorting all accept callable arguments.

predictions = [
    {"id": "P001", "score": 0.95},
    {"id": "P002", "score": 0.42},
    {"id": "P003", "score": 0.88}
]

# sorted() with lambda key
ranked = sorted(predictions, key=lambda p: p["score"], reverse=True)
print("Ranked:", [p["id"] for p in ranked])

# map() to transform — returns an iterator, wrap in list()
scores = list(map(lambda p: p["score"], predictions))
print(f"Scores: {scores}")

# filter() to select — returns an iterator
high = list(filter(lambda p: p["score"] >= 0.5, predictions))
print(f"High confidence: {[p['id'] for p in high]}")

# Common mistake: lambdas can only have ONE expression.
# If you need multiple lines, use a regular def.

# You Should See:
# Ranked: ['P001', 'P003', 'P002']
# Scores: [0.95, 0.42, 0.88]
# High confidence: ['P001', 'P003']

## 4. Classes

Classes bundle data and behavior. Every sklearn model, every FastAPI app, every PyTorch module is a class. Understanding `__init__` and `self` is non-negotiable.

In [ ]:
# WHAT: Define a class with __init__ and methods.
# WHY: When you call model.fit(X, y) in sklearn, you're calling
# a method on a class instance. Understanding this pattern unlocks
# every ML framework.

class ThresholdClassifier:
    """A simple binary classifier based on a score threshold."""
    
    def __init__(self, threshold: float = 0.5):
        # __init__ runs when you create an instance
        # self.threshold stores data ON the instance
        self.threshold = threshold
        self.is_fitted = False
    
    def fit(self, scores: list[float]):
        """Store training data statistics."""
        self.mean_score = sum(scores) / len(scores)
        self.is_fitted = True
        return self   # return self enables method chaining
    
    def predict(self, scores: list[float]) -> list[str]:
        """Classify each score."""
        if not self.is_fitted:
            raise ValueError("Must call .fit() before .predict()")
        return ["positive" if s >= self.threshold else "negative" for s in scores]

# Use it — same pattern as sklearn
clf = ThresholdClassifier(threshold=0.7)
clf.fit([0.8, 0.6, 0.9, 0.3])
results = clf.predict([0.85, 0.45, 0.72])

print(f"Threshold: {clf.threshold}")
print(f"Mean score: {clf.mean_score}")
print(f"Predictions: {results}")

# You Should See:
# Threshold: 0.7
# Mean score: 0.65
# Predictions: ['positive', 'negative', 'positive']

## 5. Dataclasses

For data containers (configs, results, records), `@dataclass` eliminates boilerplate. It auto-generates `__init__`, `__repr__`, and `__eq__`.

In [ ]:
# WHAT: Use @dataclass for clean config and result objects.
# WHY: Replaces 15 lines of __init__ boilerplate with 5 lines.
# Used heavily in MLflow, Hydra config, and FastAPI.

from dataclasses import dataclass, field
from typing import Optional

@dataclass
class TrainingConfig:
    model_name: str
    learning_rate: float = 0.001
    batch_size: int = 32
    epochs: int = 10
    gpu_enabled: bool = False
    tags: list[str] = field(default_factory=list)

# Create with partial overrides
config = TrainingConfig(
    model_name="bert-base",
    learning_rate=0.0001,
    tags=["experiment", "v3"]
)

print(config)
print(f"\nLR: {config.learning_rate}")
print(f"Tags: {config.tags}")

# Common mistake: using a mutable default like tags: list = []
# causes ALL instances to share the same list.
# Always use field(default_factory=list) for mutable defaults.

# You Should See:
# TrainingConfig(model_name='bert-base', learning_rate=0.0001, batch_size=32, epochs=10, gpu_enabled=False, tags=['experiment', 'v3'])
# LR: 0.0001
# Tags: ['experiment', 'v3']

In [ ]:
# WHAT: Dataclass with methods — data + behavior.
# WHY: Result objects that know how to format themselves
# for logging, serialization, or comparison.

from dataclasses import dataclass

@dataclass
class PredictionResult:
    model: str
    accuracy: float
    latency_ms: float
    
    def meets_sla(self, min_accuracy: float = 0.90, max_latency: float = 100) -> bool:
        """Check if this result meets production SLA requirements."""
        return self.accuracy >= min_accuracy and self.latency_ms <= max_latency
    
    def to_log_line(self) -> str:
        status = "PASS" if self.meets_sla() else "FAIL"
        return f"[{status}] {self.model}: acc={self.accuracy:.3f}, lat={self.latency_ms:.0f}ms"

results = [
    PredictionResult("bert-v2", 0.94, 45),
    PredictionResult("bert-v1", 0.88, 120),
    PredictionResult("xgboost", 0.91, 8)
]

for r in results:
    print(r.to_log_line())

# You Should See:
# [PASS] bert-v2: acc=0.940, lat=45ms
# [FAIL] bert-v1: acc=0.880, lat=120ms
# [PASS] xgboost: acc=0.910, lat=8ms

## 6. Inheritance — The .fit()/.predict() Pattern

Every sklearn estimator inherits from `BaseEstimator`. Every PyTorch model inherits from `nn.Module`. Understanding inheritance explains WHY the API is consistent.

In [ ]:
# WHAT: Create a base class and two implementations.
# WHY: This is exactly how sklearn's estimator API works.
# All models share .fit() and .predict() because they inherit
# from a common base.

from abc import ABC, abstractmethod

class BaseTransformer(ABC):
    """Base class for data transformers — like sklearn's TransformerMixin."""
    
    @abstractmethod
    def fit(self, data: list[float]) -> 'BaseTransformer':
        pass
    
    @abstractmethod
    def transform(self, data: list[float]) -> list[float]:
        pass

class MinMaxScaler(BaseTransformer):
    """Scale values to [0, 1] range."""
    
    def fit(self, data):
        self.min_val = min(data)
        self.max_val = max(data)
        return self
    
    def transform(self, data):
        rng = self.max_val - self.min_val
        return [(x - self.min_val) / rng for x in data]

class ZScoreScaler(BaseTransformer):
    """Standardize to mean=0, std=1."""
    
    def fit(self, data):
        self.mean = sum(data) / len(data)
        self.std = (sum((x - self.mean) ** 2 for x in data) / len(data)) ** 0.5
        return self
    
    def transform(self, data):
        return [(x - self.mean) / self.std for x in data]

# Both share the same API — interchangeable
raw_data = [100, 200, 300, 400, 500]

for ScalerClass in [MinMaxScaler, ZScoreScaler]:
    scaler = ScalerClass()
    scaler.fit(raw_data)
    result = scaler.transform(raw_data)
    print(f"{ScalerClass.__name__}: {[round(x, 3) for x in result]}")

# You Should See:
# MinMaxScaler: [0.0, 0.25, 0.5, 0.75, 1.0]
# ZScoreScaler: [-1.414, -0.707, 0.0, 0.707, 1.414]

In [ ]:
# WHAT: Use isinstance() to check types at runtime.
# WHY: Pipeline code often needs to handle different object types.
# isinstance() respects inheritance — preferred over type().

scaler = MinMaxScaler().fit([1, 2, 3])

print(f"Is MinMaxScaler: {isinstance(scaler, MinMaxScaler)}")
print(f"Is BaseTransformer: {isinstance(scaler, BaseTransformer)}")
print(f"Is ZScoreScaler: {isinstance(scaler, ZScoreScaler)}")

# You Should See:
# Is MinMaxScaler: True
# Is BaseTransformer: True
# Is ZScoreScaler: False

## 7. Putting It Together — A Mini Pipeline

Combine functions, classes, and dataclasses into a realistic pattern.

In [ ]:
# WHAT: Build a simple prediction pipeline using everything above.
# WHY: This is the pattern you will see in production ML services.

from dataclasses import dataclass
from typing import Optional

@dataclass
class PipelineConfig:
    threshold: float = 0.5
    min_confidence: float = 0.3
    model_version: str = "v1"

def preprocess(records: list[dict]) -> list[dict]:
    """Filter out low-quality records."""
    return [r for r in records if r.get("score") is not None]

def classify(records: list[dict], config: PipelineConfig) -> list[dict]:
    """Add classification labels based on config."""
    results = []
    for r in records:
        if r["score"] < config.min_confidence:
            label = "uncertain"
        elif r["score"] >= config.threshold:
            label = "positive"
        else:
            label = "negative"
        results.append({**r, "label": label, "model": config.model_version})
    return results

# Run the pipeline
raw_data = [
    {"id": 1, "score": 0.92},
    {"id": 2, "score": None},      # bad record
    {"id": 3, "score": 0.15},      # uncertain
    {"id": 4, "score": 0.67}
]

config = PipelineConfig(threshold=0.7, model_version="v2")
clean = preprocess(raw_data)
results = classify(clean, config)

print(f"Input: {len(raw_data)} records")
print(f"After preprocess: {len(clean)} records")
print(f"\nResults:")
for r in results:
    print(f"  ID {r['id']}: score={r['score']:.2f}, label={r['label']}, model={r['model']}")

# You Should See:
# Input: 4 records
# After preprocess: 3 records
# Results:
#   ID 1: score=0.92, label=positive, model=v2
#   ID 3: score=0.15, label=uncertain, model=v2
#   ID 4: score=0.67, label=negative, model=v2

## Recap

| Concept | Key Takeaway |
|---|---|
| `def` | Functions are first-class objects — pass them as arguments |
| Multiple returns | Returned as tuples — unpack with `a, b = func()` |
| `*args, **kwargs` | Accept arbitrary arguments — used in wrappers and logging |
| Type hints | Document intent, enable IDE support — not enforced at runtime |
| Lambda | One-line anonymous functions for `sorted()`, `map()`, `filter()` |
| Classes | Bundle data + behavior — `__init__` + `self` pattern |
| `@dataclass` | Auto-generate `__init__`, `__repr__`, `__eq__` — use for data containers |
| Inheritance | Why sklearn's `.fit()` / `.predict()` API is consistent |

**Next notebook:** [File I/O and Data Formats](Python_File_IO.ipynb)

**Full explanation:** [Functions and Classes Chapter](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/python/05_Control_Flow_Functions.md)